# 1. Algorytm genetyczny: selekcja cech

**Cel:** wybrać podzbiór cech klasyfikatora, uwzględniając jakość predykcji i koszt liczby cech. Czas: 15–20 min. Wymaga `numpy`, `scikit-learn`.

**Uruchamianie:** wykonuj komórki kolejno. Dane są generowane lokalnie; nie jest potrzebny internet.

### Model i funkcja celu
Chromosom to 10 bitów. Wynik to średnia accuracy z walidacji krzyżowej minus kara 0,02 za każdą wybraną cechę. Skalowanie odbywa się wewnątrz każdego folda (bez wycieku danych).

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X,y=make_classification(n_samples=180,n_features=10,n_informative=3,n_redundant=2,
                        random_state=17,shuffle=False)
cv=StratifiedKFold(3,shuffle=True,random_state=17)
def score(mask):
    if not np.any(mask): return -1.0
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=300,random_state=17))
    quality=cross_val_score(model,X[:,mask],y,cv=cv,scoring='accuracy').mean()
    return float(quality-0.02*mask.sum())
print('Wszystkie cechy:',round(score(np.ones(10,dtype=bool)),3))

### Selekcja, krzyżowanie, mutacja
Selekcja turniejowa wybiera lepszy osobnik z dwóch losowych kandydatów. Elita przechodzi do następnego pokolenia. Wynik cache zapobiega ponownym obliczeniom tego samego maskowania.

In [ ]:
from functools import lru_cache
rng=np.random.default_rng(17)
@lru_cache(None)
def fitness(bits): return score(np.array(bits,dtype=bool))
population=rng.integers(0,2,size=(12,10),dtype=np.int8)
def tournament(pop):
    i,j=rng.choice(len(pop),size=2,replace=False)
    return pop[i if fitness(tuple(pop[i]))>=fitness(tuple(pop[j])) else j]
history=[]
for generation in range(8):
    population=sorted(population,key=lambda a:fitness(tuple(a)),reverse=True)
    history.append(fitness(tuple(population[0])))
    elite=population[0].copy()
    children=[elite]
    while len(children)<12:
        a,b=tournament(population),tournament(population)
        cut=int(rng.integers(1,10))
        child=np.r_[a[:cut],b[cut:]].astype(np.int8)
        child^=(rng.random(10)<0.06).astype(np.int8)
        children.append(child)
    population=np.array(children)
best=max(population,key=lambda a:fitness(tuple(a)))
print('Historia:',np.round(history,3))
print('Maska:',best,'liczba cech:',best.sum(),'fitness:',round(fitness(tuple(best)),3))
assert 1<=best.sum()<=10
assert history[-1]>=history[0]


**Analiza:** Jak kara 0,02 zmienia liczbę cech? Zmień ją na 0,05 i porównaj wyniki. Walidacja krzyżowa służy tutaj do poszukiwania; końcową jakość na nowych danych trzeba sprawdzić na osobnym zbiorze testowym.